In [ ]:
import os
import numpy as np
import bigfish
import bigfish.stack as stack
import bigfish.multistack as multistack
import bigfish.plot as plot
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import dask.array as da
import seaborn as sns

In [ ]:
# Load the data
loc = r'/Users/ericron/Desktop/uqbio24/DUSP1_conc_sweep_R2_0min_071422.h5'
h5_file = h5py.File(loc, 'r') # making this read is important otherwise you may get WINERROR 33

In [ ]:
# Print the groups in the file
level = 1

def print_group(name, obj):
    # Calculate current level based on the name depth
    current_level = name.count('/')
    if current_level <= level:
        print("  " * 2 * current_level + f"- {name}")

h5_file.visititems(print_group)

In [ ]:
def inspect_specific_keys(h5_file, keys_to_check):
    """
    Inspect specified keys in an HDF5 file, including their shapes and data types.
    
    Parameters:
    - h5_file: h5py.File, the loaded HDF5 file object.
    - keys_to_check: list of str, the specific keys to inspect.
    """
    for key in keys_to_check:
        if key in h5_file:
            obj = h5_file[key]
            if isinstance(obj, h5py.Group):
                print(f"Group: {key}/")
            elif isinstance(obj, h5py.Dataset):
                print(f"Dataset: {key}")
                print(f"  Shape: {obj.shape}")
                print(f"  Type: {obj.dtype}")
            else:
                print(f"Unknown item: {key}")
        else:
            print(f"Key not found: {key}")


# Define the keys to inspect
keys_to_check = [
    "Analysis_ER_Dec0324_2024-12-11/bigfish_threshold",
    "Analysis_ER_Dec0324_2024-12-11/df_cellresults",
    "Analysis_ER_Dec0324_2024-12-11/df_clusterresults",
    "Analysis_ER_Dec0324_2024-12-11/df_spotresults",
    "Analysis_ER_Dec0324_2024-12-11/parameters",
    "masks",
    "metadata",
    "raw_images",
]

# Inspect the specified keys
inspect_specific_keys(h5_file, keys_to_check)


In [ ]:
# Load the data
masks = np.array(h5_file['masks'])
print('Masks shape:', masks.shape)
print('Masks dtype:', masks.dtype)

images = np.array(h5_file['raw_images'])
print('Images shape:', images.shape)
print('Images dtype:', images.dtype)

theshold = np.array(h5_file['Analysis_ER_Dec0324_2024-12-11/bigfish_threshold'])
spots = pd.read_hdf(loc, key ='Analysis_ER_Dec0324_2024-12-11/df_spotresults')
clusters = pd.read_hdf(loc, key ='Analysis_ER_Dec0324_2024-12-11/df_clusterresults')
cell_results = pd.read_hdf(loc, key ='Analysis_ER_Dec0324_2024-12-11/df_cellresults')

In [ ]:
# Separate the masks
cyto_masks = masks[:,:,1,0,:,:].squeeze()
nuclei_masks = masks[:,:,2,0,:,:].squeeze()
print('Cyto masks shape:', cyto_masks.shape)
print('Nuclei masks shape:', nuclei_masks.shape)

In [ ]:
# rescale the images
images = images.squeeze()
images_rescaled = stack.rescale(images, channel_to_stretch=[0,1,2])
print('Images rescaled shape:', images_rescaled.shape)


In [ ]:
FISH_mip = np.max(images_rescaled[:,0,:,:,:], axis=1)
print('FISH_mip shape:', FISH_mip.shape)

cyto_mip = np.max(images_rescaled[:,1,:,:,:], axis=1)
print('Cyto_mip shape:', cyto_mip.shape)

nuclei_mip = np.max(images_rescaled[:,2,:,:,:], axis=1)
print('Nuclei_mip shape:', nuclei_mip.shape)

In [ ]:
# Plot cyto_masks, nuclei_masks, cyto_mip, nuclei_mip for each cell 

# List of images and their titles
image_types = [cyto_masks, cyto_mip, nuclei_masks, nuclei_mip]
titles = ["Cyto Mask", "Cyto MIP", "Nuclei Mask", "Nuclei MIP"]

# Loop through each cell
n_cells = cyto_masks.shape[0]  # Number of cells
for cell_idx in range(n_cells):
    # Create a new figure for each cell
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))  # Adjust figsize to be less wide if needed
    fig.suptitle(f"Cell {cell_idx}", fontsize=16)
    
    for img_idx, (image, title) in enumerate(zip(image_types, titles)):
        ax = axes[img_idx]
        ax.imshow(image[cell_idx], cmap="cividis")
        ax.set_title(title)
        ax.axis("off")
    
    # Reduce spacing between subplots
    plt.subplots_adjust(wspace=0.05)  # Adjust horizontal spacing
    plt.tight_layout(pad=0.5)  # Reduce padding between figure and plots
    
    plt.show()


In [ ]:
spots.head()



In [ ]:
cell_results.head()